In [1]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from desmin_reconstruction.plotting import save_panel
from desmin_reconstruction.utils.io import read_SIM_data
from IPython.display import display, SVG
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea
from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter


COLOR_DICT = {
    "desmin": "#CC3311",
    "actin": "#0077BB",
    "actinin": "#009988",
    "myosin": "#BBBBBB",
}
LABEL_DICT = {
    "desmin": "Desmin",
    "actin": "Actin",
    "actinin": "α-actinin",
    "myosin": "Myosin",
}

datadir = Path("../../data/SIM")


@dataclass(frozen=True)
class SIMPanelSpec:
    protein: str
    path: str
    desmin_channel: int
    partner_channel: int
    point_voi: tuple | None = None


DEFAULT_SPECS = (
    SIMPanelSpec(
        "actin", "actin/Image10_actin_SIM2.h5", 1, 0, (512, 1536, 512, 1536, 0, 45)
    ),
    SIMPanelSpec(
        "actinin", "alpha-actinin/Image2_SIM2.h5", 0, 1, (1024, 2048, 0, 1024, 0, 45)
    ),
    SIMPanelSpec("myosin", "myosin/Image20_myosin_SIM2.h5", 0, 1, None),
)

In [ ]:
@dataclass
class PreparedSIM:
    protein: str
    grids: dict
    point_voi: tuple


@dataclass
class RenderedSIM:
    protein: str
    image: np.ndarray
    extent: tuple  # Local physical xmin, xmax, ymin, ymax in um.
    image_extent: tuple  # Actual local camera extent after pixel rounding.
    world_bounds: tuple  # Original cropped XYZ coordinates, in um.
    colors: dict


def prepare_sim_pair(
    data,
    pixel_sizes,
    *,
    protein,
    desmin_channel,
    partner_channel,
    point_voi=None,
    spacing_scale=1.0,
):
    """Prepare two registered cell-data grids from channel arrays in Z,Y,X.

    pixel_sizes is (dz, dy, dx). It must be in micrometers after multiplying
    by spacing_scale; e.g. use 0.001 only if the reader returns nanometers.

    point_voi uses PyVista extract_subset point-index bounds:
        (xmin, xmax, ymin, ymax, zmin, zmax)
    For CELL data this is equivalent to array[zmin:zmax, ymin:ymax,
    xmin:xmax]. Upper array slice bounds are exclusive. Cropping before
    grid construction avoids allocating two full-size VTK grids.

    Array orientation, intensity values and physical spacing are retained.
    The unused roi=(slice(...), ...) assignments in the original code are
    deliberately not applied; the point_voi crops were the ones plotted.
    """
    import pyvista as pv

    spacing = np.asarray(pixel_sizes, dtype=float) * float(spacing_scale)
    if spacing.shape != (3,) or not np.isfinite(spacing).all() or np.any(spacing <= 0):
        raise ValueError(
            "pixel_sizes * spacing_scale must be positive finite (dz, dy, dx)."
        )
    if protein == "desmin" or desmin_channel == partner_channel:
        raise ValueError(
            "Choose a distinct partner protein and two different channels."
        )
    volumes = {
        "desmin": np.asarray(data[desmin_channel]),
        protein: np.asarray(data[partner_channel]),
    }
    shape = volumes["desmin"].shape
    if len(shape) != 3 or any(n < 1 for n in shape):
        raise ValueError("Each channel must be a nonempty 3D array in Z,Y,X order.")
    if volumes[protein].shape != shape:
        raise ValueError("The two channels must have matching registered shapes.")
    nz, ny, nx = shape
    if point_voi is None:
        voi = (0, nx, 0, ny, 0, nz)
    else:
        raw = np.asarray(point_voi, dtype=float)
        if (
            raw.shape != (6,)
            or not np.isfinite(raw).all()
            or np.any(raw != np.floor(raw))
        ):
            raise ValueError("point_voi must contain six integer point indices.")
        voi = tuple(int(v) for v in raw)
    x0, x1, y0, y1, z0, z1 = voi
    if not (0 <= x0 < x1 <= nx and 0 <= y0 < y1 <= ny and 0 <= z0 < z1 <= nz):
        raise ValueError(
            f"point_voi {voi} is outside cell dimensions (nx,ny,nz)={(nx, ny, nz)}."
        )
    origin = np.array([x0, y0, z0]) * spacing[::-1]
    grids = {}
    for name, array in volumes.items():
        cropped = array[z0:z1, y0:y1, x0:x1]
        if (
            not np.issubdtype(cropped.dtype, np.number)
            or not np.isfinite(cropped).all()
        ):
            raise ValueError(f"{name}: expected finite numeric image intensities.")
        grid = pv.ImageData(
            dimensions=np.array(cropped.shape[::-1]) + 1,
            spacing=spacing[::-1],
            origin=origin,
        )
        grid.cell_data[name] = cropped.transpose(2, 1, 0).ravel(order="F")
        grids[name] = grid
    return PreparedSIM(protein, grids, voi)


def load_sim_pair(path, read_SIM_data, **prep_kwargs):
    """Use the experiment's existing HDF5 reader, then the common prep step."""
    data, pixel_sizes = read_SIM_data(Path(path))
    return prepare_sim_pair(data, pixel_sizes, **prep_kwargs)


def render_sim_pair(
    prepared,
    *,
    colors=None,
    height_px=1400,
    clim=(0.0, 0.2),
    opacity=(0.0, 0.1, 0.8),
    color_modes=None,
):
    """Render both channels together using the original volume settings.

    By default desmin/actinin use constant hue and actin/myosin use a
    black-to-color ramp, matching the supplied code's transfer functions.
    Override explicitly with e.g. color_modes={'actin': 'constant'}.
    clim and opacity are shared by both channels. Both use maximum blending
    and linear volume interpolation. This is a PyVista two-volume scene,
    not a separately computed NumPy projection or a blend of two screenshots.
    """
    import pyvista as pv

    colors = {**COLOR_DICT, **(colors or {})}
    modes = {
        "desmin": "constant",
        "actin": "ramp",
        "actinin": "constant",
        "myosin": "ramp",
        **(color_modes or {}),
    }
    height_px = int(height_px)
    if height_px < 2:
        raise ValueError("height_px must be at least 2.")
    limits = np.asarray(clim, dtype=float)
    if limits.shape != (2,) or not np.isfinite(limits).all() or limits[0] >= limits[1]:
        raise ValueError("clim must have two increasing finite values.")
    opacity = np.asarray(opacity, dtype=float)
    if (
        opacity.ndim != 1
        or len(opacity) < 2
        or not np.isfinite(opacity).all()
        or np.any((opacity < 0) | (opacity > 1))
    ):
        raise ValueError("opacity must contain at least two values between 0 and 1.")
    bounds = np.asarray(prepared.grids["desmin"].bounds, dtype=float)
    x0, x1, y0, y1, z0, z1 = bounds
    dx, dy = x1 - x0, y1 - y0
    width_px = max(2, int(np.ceil(height_px * dx / dy)))
    cx, cy, cz = (x0 + x1) / 2, (y0 + y1) / 2, (z0 + z1) / 2
    distance = 2 * max(dx, dy, z1 - z0)
    half_width = (dy / 2) * width_px / height_px
    p = pv.Plotter(off_screen=True, window_size=(width_px, height_px))
    try:
        for name in ("desmin", prepared.protein):
            mode = modes.get(name, "constant")
            if mode == "constant":
                cmap = [colors[name]]
            elif mode == "ramp":
                cmap = LinearSegmentedColormap.from_list(
                    f"{name}_ramp", ["black", colors[name]]
                )
            else:
                raise ValueError("Each color mode must be 'constant' or 'ramp'.")
            actor = p.add_volume(
                prepared.grids[name],
                scalars=name,
                cmap=cmap,
                opacity=opacity.tolist(),
                clim=tuple(limits),
                blending="maximum",
                show_scalar_bar=False,
                reset_camera=False,
            )
            actor.prop.interpolation_type = "linear"
        p.set_background("black")
        p.enable_parallel_projection()
        p.camera_position = [(cx, cy, cz + distance), (cx, cy, cz), (0, 1, 0)]
        p.camera.parallel_scale = dy / 2
        p.reset_camera_clipping_range()
        rgb = np.array(p.screenshot(transparent_background=False), copy=True)
    finally:
        p.close()
    # Local labels translate the cropped origin to zero; the scale is retained.
    return RenderedSIM(
        prepared.protein,
        rgb,
        (0.0, dx, 0.0, dy),
        (cx - half_width - x0, cx + half_width - x0, 0.0, dy),
        tuple(bounds),
        colors,
    )


def render_sim_row(
    datadir,
    read_SIM_data,
    *,
    specs=DEFAULT_SPECS,
    spacing_scale=1.0,
    colors=None,
    height_px=1400,
    clim=(0.0, 0.2),
    opacity=(0.0, 0.1, 0.8),
    color_modes=None,
):
    """Load, prepare and render one experiment at a time to limit memory use."""
    views = []
    for spec in specs:
        prepared = load_sim_pair(
            Path(datadir) / spec.path,
            read_SIM_data,
            protein=spec.protein,
            desmin_channel=spec.desmin_channel,
            partner_channel=spec.partner_channel,
            point_voi=spec.point_voi,
            spacing_scale=spacing_scale,
        )
        views.append(
            render_sim_pair(
                prepared,
                colors=colors,
                height_px=height_px,
                clim=clim,
                opacity=opacity,
                color_modes=color_modes,
            )
        )
        del prepared
    return tuple(views)


def _pair_title(ax, protein, colors, fontsize, pad):
    parts = [
        TextArea(
            LABEL_DICT.get(protein, protein),
            textprops={"color": colors[protein], "fontsize": fontsize},
        ),
        TextArea(" / ", textprops={"color": "white", "fontsize": fontsize}),
        TextArea("Desmin", textprops={"color": colors["desmin"], "fontsize": fontsize}),
    ]
    box = HPacker(children=parts, align="center", pad=0, sep=0)
    header = AnchoredOffsetbox(
        loc="lower center",
        child=box,
        frameon=False,
        pad=0,
        borderpad=pad / fontsize,
        bbox_to_anchor=(0.5, 1),
        bbox_transform=ax.transAxes,
        prop={"size": fontsize},
    )
    header.set_clip_on(False)
    ax.add_artist(header)


def compose_sim_row(
    rendered,
    *,
    panel_height=2.6,
    panel_gap=0.32,
    figsize=None,
    margins=None,
    rois=(0.0, 0.0, 4.0, 4.0),
    roi_linewidth=0.9,
    roi_linestyle=(0, (3, 2)),
    xticks=None,
    yticks=None,
    tick_format="%.1f",
    ytick_rotation=90,
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    xtick_fontsize=10,
    ytick_fontsize=10,
    title_fontsize=14,
    title_pad=7,
    xtick_pad=5,
    ytick_pad=7,
    xlabel_pad=6,
    ylabel_pad=7,
    spine_color="#AAAAAA",
    font_family="DejaVu Sans",
    share_y="auto",
):
    """Assemble screenshots and vector annotations, matching the supplied row.

    panel_height, panel_gap, figsize and margins are in INCHES. Fonts, ROI
    linewidth and all *_pad values are in POINTS. margins=(left,right,bottom,top).
    figsize=None sizes the canvas from panel_height. Supplying figsize fits
    the panels inside it while preserving each image's physical aspect ratio.

    rois: (left,bottom,width,height) applies to every panel, or a dictionary
    keyed by protein supplies individual rectangles. None hides all boxes;
    an omitted dictionary key or None value hides that panel's box. Coordinates
    are local micrometers after the image's cropped origin is translated to 0.

    Axis limits come from calibrated grid spacing. Equal Y ranges share an
    axis automatically; different ranges keep numeric Y labels on all panels.
    share_y=True requires equal Y ranges. X labels remain centered.
    """
    rendered = tuple(rendered)
    if not rendered:
        raise ValueError("Supply at least one rendered panel.")
    if panel_height <= 0 or panel_gap < 0:
        raise ValueError("panel_height must be positive and panel_gap nonnegative.")
    if share_y not in (True, False, "auto"):
        raise ValueError("share_y must be True, False or 'auto'.")
    same_y = all(
        np.allclose(v.extent[2:], rendered[0].extent[2:], rtol=1e-6, atol=1e-9)
        for v in rendered
    )
    if share_y is True and not same_y:
        raise ValueError(
            "Y ranges differ; use share_y='auto' to retain their calibrated scales."
        )
    share_y = same_y if share_y == "auto" else share_y
    aspects = np.array(
        [(v.extent[1] - v.extent[0]) / (v.extent[3] - v.extent[2]) for v in rendered]
    )
    if margins is None:
        left = max(
            0.65,
            (ytick_pad + ylabel_pad + 1.3 * (ytick_fontsize + ylabel_fontsize) + 8)
            / 72,
        )
        bottom = max(
            0.60,
            (xtick_pad + xlabel_pad + 1.3 * (xtick_fontsize + xlabel_fontsize) + 8)
            / 72,
        )
        right = max(0.18, 1.5 * xtick_fontsize / 72)
        top = max(0.38, (1.4 * title_fontsize + title_pad + 7) / 72)
    else:
        left, right, bottom, top = map(float, margins)
        if min(left, right, bottom, top) < 0:
            raise ValueError("Margins must be nonnegative.")
    gaps = (len(rendered) - 1) * panel_gap
    if figsize is None:
        widths = panel_height * aspects
        fw = left + widths.sum() + gaps + right
        fh = bottom + panel_height + top
        start_x, start_y = left, bottom
    else:
        fw, fh = map(float, figsize)
        available_w, available_h = fw - left - right - gaps, fh - bottom - top
        if min(available_w, available_h) <= 0:
            raise ValueError("figsize is too small for the margins and panel gaps.")
        panel_height = min(available_h, available_w / aspects.sum())
        widths = panel_height * aspects
        start_x = left + (available_w - widths.sum()) / 2
        start_y = bottom + (available_h - panel_height) / 2
    total_width = widths.sum() + gaps
    axes = []
    with (
        plt.style.context("default"),
        mpl.rc_context(
            {
                "font.family": font_family,
                "figure.autolayout": False,
                "figure.constrained_layout.use": False,
                "text.usetex": False,
            }
        ),
    ):
        fig = plt.figure(figsize=(fw, fh), facecolor="black")
        x = start_x
        for i, (view, width) in enumerate(zip(rendered, widths)):
            shared = {"sharey": axes[0]} if i and share_y else {}
            ax = fig.add_axes(
                [x / fw, start_y / fh, width / fw, panel_height / fh],
                facecolor="black",
                **shared,
            )
            axes.append(ax)
            x0, x1, y0, y1 = view.extent
            ax.imshow(
                view.image,
                extent=view.image_extent,
                origin="upper",
                interpolation="none",
                aspect="equal",
            )
            tx = np.linspace(x0, x1, 5) if xticks is None else np.asarray(xticks)
            ty = np.linspace(y0, y1, 5) if yticks is None else np.asarray(yticks)
            ax.set_xticks(tx)
            ax.set_yticks(ty)
            ax.set_xlim(x0, x1)
            ax.set_ylim(y0, y1)
            ax.xaxis.set_major_formatter(FormatStrFormatter(tick_format))
            ax.yaxis.set_major_formatter(FormatStrFormatter(tick_format))
            ax.tick_params(colors="white", direction="in", length=3, width=0.7)
            ax.tick_params(axis="x", labelsize=xtick_fontsize, pad=xtick_pad)
            ax.tick_params(
                axis="y",
                left=True,
                labelleft=(i == 0 or not share_y),
                labelsize=ytick_fontsize,
                pad=ytick_pad,
                labelrotation=ytick_rotation,
            )
            for side, spine in ax.spines.items():
                spine.set_visible(side in ("left", "bottom"))
                spine.set_color(spine_color)
                spine.set_linewidth(0.7)
            for label in ax.get_xticklabels():
                label.set_ha("center")
            for value, label in zip(ax.get_yticks(), ax.get_yticklabels()):
                label.set_va("bottom" if np.isclose(value, y0) else "center")
            _pair_title(ax, view.protein, view.colors, title_fontsize, title_pad)
            roi = rois.get(view.protein) if isinstance(rois, dict) else rois
            if roi is not None:
                rx, ry, rw, rh = map(float, roi)
                if not np.isfinite([rx, ry, rw, rh]).all() or rw <= 0 or rh <= 0:
                    raise ValueError(
                        """Each ROI must have finite coordinates and positive 
                        width/height."""
                    )
                ax.add_patch(
                    Rectangle(
                        (rx, ry),
                        rw,
                        rh,
                        fill=False,
                        edgecolor="white",
                        linewidth=roi_linewidth,
                        linestyle=roi_linestyle,
                        clip_on=False,
                        zorder=5,
                    )
                )
            x += width + panel_gap
        # Measure actual tick text so the shared labels have useful point padding.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        xboxes = [
            t.get_window_extent(renderer)
            for ax in axes
            for t in ax.get_xticklabels()
            if t.get_visible()
        ]
        yboxes = [
            t.get_window_extent(renderer)
            for t in axes[0].get_yticklabels()
            if t.get_visible()
        ]
        if xboxes:
            xlabel_y = min(b.y0 for b in xboxes) / fig.bbox.height - xlabel_pad / (
                72 * fh
            )
        else:
            xlabel_y = start_y / fh - xlabel_pad / (72 * fh)
        if yboxes:
            ylabel_x = min(b.x0 for b in yboxes) / fig.bbox.width - ylabel_pad / (
                72 * fw
            )
        else:
            ylabel_x = start_x / fw - ylabel_pad / (72 * fw)
        fig.supxlabel(
            "x (μm)",
            x=(start_x + total_width / 2) / fw,
            y=xlabel_y,
            va="top",
            color="white",
            fontsize=xlabel_fontsize,
        )
        fig.supylabel(
            "y (μm)",
            x=ylabel_x,
            y=(start_y + panel_height / 2) / fh,
            ha="right",
            color="white",
            fontsize=ylabel_fontsize,
        )
    return fig, tuple(axes)

In [9]:
rendered = render_sim_row(
    datadir,
    read_SIM_data,
    colors=COLOR_DICT,
    height_px=1400,
    clim=(0.0, 0.2),
    opacity=(0.0, 0.1, 0.8),
    color_modes={
        "desmin": "constant",
        "actin": "constant",
        "actinin": "constant",
        "myosin": "constant",
    },
)

# 2. Assemble the figure.
# Rerun this section to change layout without rerendering.
fig, axes = compose_sim_row(
    rendered,
    panel_height=2.6,  # Inches
    panel_gap=0.32,  # Inches
    # Each rectangle: left, bottom, width, height in local µm.
    rois={
        "actin": (0, 0, 4, 4),
        "actinin": (0, 0, 4, 4),
        "myosin": (0, 0, 4, 4),
    },
    roi_linewidth=0.9,
    roi_linestyle=(0, (3, 2)),
    title_fontsize=14,
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    xtick_fontsize=10,
    ytick_fontsize=10,
    ytick_rotation=90,
    tick_format="%.1f",
    xlabel_pad=6,  # Points; smaller brings label closer
    ylabel_pad=7,
)

In [10]:
savepath = "../../figures/3e"
save_panel(fig, savepath, formats=("svg",))
display(SVG(filename=str(savepath + ".svg")))
plt.close(fig)